In [1]:
%cd /home/../datadrive/MMMM

/datadrive/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")
import time

# Import Pytorch
import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter

# Import Pretrain Libraries (transformers + diffusers)
from transformers import BartTokenizer
from diffusers import AutoencoderKL

# Parallel Helper
# from parallel import DataParallelModel, DataParallelCriterion

# Import Our Own Functions
from master_init import *
from DSG import *

from count_params import count_params

# Import Tasks
import EEG_TEXT_BART
import EEG_TEXT_BART_SENTIMENT
import EEG_IMG_DIFFUSION
import EEG_IMG_CLASSIFICATION
import PRETRAIN_EEG_IMG_CLIP_MATCHING
import PRETRAIN_EEG_TEXT_CLIP_MATCHING
import PRETRAIN_EEG_IMG_UNET
import PRETRAIN_EEG_TEXT_UNET

In [3]:
torch.set_default_dtype(torch.float32)

# config = get_config() Implement later
print(f"[INFO] FETCHING CONFIGURATIONS.")
config = {
    "device" : "cuda",
    "device_ids" : [0,1,2,3],
    "staging_device" : "cuda",
    "num_epochs" : 100,
    "use_non_pytorch_parallel" : False,
    "test_run" : True,
    "live_evaluate" : True,
    "eval_interval" : 1,
    "log_dir" : "./logs"
}

device = config["device"]
device_ids = config["device_ids"]
staging_device = config["staging_device"]
num_epochs = config["num_epochs"]
use_non_pytorch_parallel = config["use_non_pytorch_parallel"]
test_run = config["test_run"]
live_evaluate = config["live_evaluate"]
eval_interval = config["eval_interval"]
log_dir = config["log_dir"]

print(f"[INFO] FINISHED CONFIGURATIONS.")

[INFO] FETCHING CONFIGURATIONS.
[INFO] FINISHED CONFIGURATIONS.


In [4]:
print(f"[INFO] INITIALIZING MODEL.")
model = INITIALIZE_MODEL(device=None, device_ids=device_ids, dtype=torch.float32)
if use_non_pytorch_parallel:
    model = DataParallelModel(model, device_ids=device_ids).to(device)
else:
    model = nn.DataParallel(model, device_ids=device_ids).to(device)

[INFO] INITIALIZING MODEL.
LOADED EEG ENCODER
LOADED CLIP ENCODER
LOADED BART MODEL
LOADED EEG-TEXT-BART
LOADED U-NET
LOADED CLIP TOKENIZER
LOADED EEG-IMG-DIFFUSION
LOADED EEG-IMG-CLASSIFICATION
LOADED EEG-TEXT-SENTIMENT


In [11]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["ZuCo-BART", "ZuCo-CLIP", "Brain2Image"],
    bsz=[256, 256, 1],
    dev_bsz=[256, 256, 1]
)
# if test_run:
#     for key, val in dataset_dict.items():
#         dataset_dict[key]["train"] = dataset_dict[key]["dev"] # Make things faster
#         dataset_dict[key]["dev"] = dataset_dict[key]["test"] # Mimic an unseen set

print(f"[INFO] PARAMETER COUNT")
print(f"[INFO] >>>> {count_params(model)} TOTAL PARAMETERS.")
print(f"[INFO] >>>> {count_params(model,True)} TRAINABLE PARAMETERS.")
print(f"[INFO] >>>> {count_params(model,False)} NON-TRAINABLE PARAMETERS.")

[INFO] PARAMETER COUNT
[INFO] >>>> 1508803952 TOTAL PARAMETERS.
[INFO] >>>> 598623916 TRAINABLE PARAMETERS.
[INFO] >>>> 910180036 NON-TRAINABLE PARAMETERS.


In [6]:
# Load Pretrains
print(f"[INFO] LOADING PRETRAINS...")
BART_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(dtype=torch.float32)
vae.requires_grad_(False)
print(f"[INFO] LOADED PRETRAINS.")

# Decision not to put dataloders into DSGTask() object
# Will take too much space and some datasets repeatedly used for
# different tasks.
print(f"[INFO] SETTING UP TRAINING TASKS...")
dsg_tasks = DSGTasks()

dsg_tasks.add_task(
    DSGTask(
        task_name="PRETRAIN-EEG-TEXT-CLIP-MATCHING",
        dataset_tag="ZuCo-CLIP",
        criterion=nn.KLDivLoss(reduction="batchmean"), # Symmetrized with Lambda inside train()
        optimizer=optim.Adam,
        learning_rate=1e-3,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

[INFO] LOADING PRETRAINS...
[INFO] LOADED PRETRAINS.
[INFO] SETTING UP TRAINING TASKS...


In [7]:
task = dsg_tasks.tasks[0]

In [13]:
args_dict = {
    "model" : model,
    "dataloader" : dataset_dict[task.dataset_tag],
    "optimizer" : task.optimizer(model.parameters(), lr=task.learning_rate),
    "tokenizer" : BART_tokenizer,
    "criterion" : task.criterion,
    "device" : device,
    "device_ids" : device_ids,
    "staging_device" : staging_device,
    "use_unet" : False,
    "dev_bsz" : 256,
    "bool_eval" : True,
    "temperature" : 25,
}
results = PRETRAIN_EEG_TEXT_CLIP_MATCHING.train(args_dict, using_non_pytorch_parallel=use_non_pytorch_parallel)

tensor([[0.0039, 0.0022, 0.0052, 0.0048, 0.0019, 0.0027, 0.0051, 0.0063],
        [0.0038, 0.0023, 0.0052, 0.0047, 0.0019, 0.0027, 0.0053, 0.0064],
        [0.0039, 0.0021, 0.0051, 0.0047, 0.0019, 0.0026, 0.0050, 0.0063],
        [0.0041, 0.0021, 0.0050, 0.0049, 0.0019, 0.0026, 0.0047, 0.0058],
        [0.0040, 0.0022, 0.0052, 0.0047, 0.0019, 0.0026, 0.0050, 0.0062],
        [0.0039, 0.0022, 0.0050, 0.0046, 0.0018, 0.0026, 0.0051, 0.0062],
        [0.0038, 0.0022, 0.0052, 0.0045, 0.0019, 0.0026, 0.0051, 0.0066],
        [0.0041, 0.0021, 0.0049, 0.0048, 0.0019, 0.0026, 0.0048, 0.0060]],
       device='cuda:0', grad_fn=<SliceBackward0>)
tensor([[8.0569e-01, 6.5032e-05, 6.2165e-05, 3.5182e-03, 8.2665e-04, 2.3013e-05,
         1.6757e-04, 3.7452e-05],
        [9.5246e-05, 4.2010e-01, 4.5412e-05, 8.4381e-05, 1.3743e-04, 8.0015e-05,
         3.1675e-03, 6.4361e-05],
        [7.0945e-07, 3.5386e-07, 9.9978e-01, 4.6580e-06, 1.6156e-06, 2.7962e-08,
         3.1184e-07, 1.6103e-07],
        [4.9

ZeroDivisionError: float division by zero

In [ ]:
model = results["model"]
if "train_accuracy" in results:
    if epoch % eval_interval == 0 and live_evaluate:
        print(f">>>>>>>> TRAIN ACCURACY: {results['train_accuracy'] * 100 : 8.4f} % DEV ACCURACY: {results['dev_accuracy'] * 100 : 8.4f} %")
        train_writer.add_scalar(f"{task.name} Accuracy", results['train_accuracy'] * 100, epoch)
        dev_writer.add_scalar(f"{task.name} Accuracy", results['dev_accuracy'] * 100, epoch)

print(f">>>> {task.name} | TRAIN: {results['train_loss']} DEV: {results['dev_loss']} TIME: {time.time() - start:.2f} SECONDS")
train_writer.add_scalar(f"{task.name} Loss", results['train_loss'], epoch)
dev_writer.add_scalar(f"{task.name} Loss", results['dev_loss'], epoch)